In [1]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/emanuel-gf/FTL-Embeddings-Drones.git"  # preencha com o link do repositório
REPO_DIR = Path("/content/webinar-DL")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pystac-client", "transformers", "tqdm", "rasterio", "geopandas", "pyyaml", "pillow"], check=True)

ROOT = Path.cwd()
assert (ROOT / "src").is_dir(), f"Pasta do projeto não encontrada em {ROOT}. Abra o notebook dentro de webinar-DL."
sys.path.insert(0, str(ROOT))
print("Pasta do projeto (ROOT):", ROOT, "| Colab:", IN_COLAB)

Pasta do projeto (ROOT): /Users/macuser/Documents/UN/projects/webinar-DL | Colab: False


In [2]:
import json
import time
from collections import Counter

import numpy as np
import pandas as pd
import pystac_client
import requests
import rasterio
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel

from src.annotations import index_with_paths
from src.tiler import tile_cogs

W1112 22:20:29.214000 39305 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
STAC_API = "https://api.imagery.hotosm.org/stac/"
COLLECTION = "openaerialmap"

# Pastas do projeto
COG_DIR = ROOT / "data" / "cog"                # imagens nativas, cerca de 5 cm por pixel
COG10_DIR = ROOT / "data" / "cog_10cm"         # mesmas imagens reamostradas para 10 cm
LABELS_DIR = COG10_DIR / "geojson_labelling"   # rótulos feitos antes da aula
STAC_DIR = ROOT / "data" / "stac"              # metadados STAC de cada item
for folder in (COG_DIR, STAC_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# IDs das imagens que vamos usar nesta aula
ITEM_IDS = [
    "68c41fddae0fc8076633a74e",
    "68a78c9e27e991e401df2876",
    "68a78cb427e991e401df28a2",
    "68a78ce127e991e401df2956",
    "68a78caf27e991e401df289b",
    "68a78d0527e991e401df29cc",
    "685740b9e319ef725278cb92",
    "685740c7e319ef725278cbcf",
    "685740ade319ef725278cb5a",
    "685740c3e319ef725278cbc1",
    "68a4a953e50784296730bf0f",
    "68a5bc60e5584a0f62bfa185",
    "68a5bc63e5584a0f62bfa195",
    "68a5bc16e5584a0f62bfa08c",
    "68a5bc04e5584a0f62bfa047",
    "68a5bc70e5584a0f62bfa1ce",
    "68a5bc31e5584a0f62bfa0ff",
]
print(f"{len(ITEM_IDS)} imagens (IDs únicos)")

17 imagens (IDs únicos)


In [4]:
client = pystac_client.Client.open(STAC_API)
collection = client.get_collection(COLLECTION)

items = {}
for item_id in tqdm(ITEM_IDS, desc="Buscando itens no STAC"):
    item = collection.get_item(item_id)
    items[item_id] = item
    (STAC_DIR / f"{item_id}.json").write_text(json.dumps(item.to_dict(), indent=2))

print(f"{len(items)} itens recuperados; JSON salvos em {STAC_DIR.relative_to(ROOT)}/")

Buscando itens no STAC:   0%|          | 0/17 [00:00<?, ?it/s]

17 itens recuperados; JSON salvos em data/stac/


In [5]:
def download_cog(url: str, dest: Path, expected_size: int | None = None) -> str:
    # Baixa um COG para o disco. Devolve 'skipped', 'downloaded' ou 'failed'.
    if dest.exists() and expected_size and dest.stat().st_size == expected_size:
        return "skipped"
    try:
        with requests.get(url, stream=True, timeout=120) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0)) or expected_size
            tmp = dest.with_suffix(".part")
            with open(tmp, "wb") as f, tqdm(
                total=total, unit="B", unit_scale=True, unit_divisor=1024,
                desc=dest.name, leave=False,
            ) as bar:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    f.write(chunk)
                    bar.update(len(chunk))
            tmp.rename(dest)
        return "downloaded"
    except requests.RequestException as err:
        print(f"Falhou {dest.name}: {err}")
        return "failed"

status = {}
for item_id, item in tqdm(items.items(), desc="Imagens"):
    asset = item.assets["visual"]
    dest = COG_DIR / f"{item_id}.tif"
    expected = asset.extra_fields.get("file:size")
    status[item_id] = download_cog(asset.href, dest, expected)

print(dict(Counter(status.values())))

Imagens:   0%|          | 0/17 [00:00<?, ?it/s]

{'skipped': 17}


In [6]:
rows = []
for item_id in ITEM_IDS:
    path = COG_DIR / f"{item_id}.tif"
    rows.append({"item_id": item_id, "existe": path.exists(),
                 "tamanho_MB": round(path.stat().st_size / 1e6, 1) if path.exists() else None})
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(f"Total em disco: {summary['tamanho_MB'].sum():.1f} MB em {COG_DIR.relative_to(ROOT)}/")

                 item_id  existe  tamanho_MB
68c41fddae0fc8076633a74e    True        21.9
68a78c9e27e991e401df2876    True         7.4
68a78cb427e991e401df28a2    True        13.8
68a78ce127e991e401df2956    True         8.8
68a78caf27e991e401df289b    True        19.8
68a78d0527e991e401df29cc    True         8.1
685740b9e319ef725278cb92    True        21.5
685740c7e319ef725278cbcf    True        11.5
685740ade319ef725278cb5a    True         8.0
685740c3e319ef725278cbc1    True         6.9
68a4a953e50784296730bf0f    True        11.6
68a5bc60e5584a0f62bfa185    True        17.1
68a5bc63e5584a0f62bfa195    True        11.8
68a5bc16e5584a0f62bfa08c    True        22.2
68a5bc04e5584a0f62bfa047    True        10.0
68a5bc70e5584a0f62bfa1ce    True        16.6
68a5bc31e5584a0f62bfa0ff    True         6.4
Total em disco: 223.4 MB em data/cog/


In [7]:
!{sys.executable} resampling.py --input data/cog --output data/cog_10cm --gsd 0.10

skipped (already exists): data/cog_10cm/685740ade319ef725278cb5a.tif
skipped (already exists): data/cog_10cm/685740b9e319ef725278cb92.tif
skipped (already exists): data/cog_10cm/685740c3e319ef725278cbc1.tif
skipped (already exists): data/cog_10cm/685740c7e319ef725278cbcf.tif
skipped (already exists): data/cog_10cm/68a4a953e50784296730bf0f.tif
skipped (already exists): data/cog_10cm/68a5bc04e5584a0f62bfa047.tif
skipped (already exists): data/cog_10cm/68a5bc16e5584a0f62bfa08c.tif
skipped (already exists): data/cog_10cm/68a5bc31e5584a0f62bfa0ff.tif
skipped (already exists): data/cog_10cm/68a5bc60e5584a0f62bfa185.tif
skipped (already exists): data/cog_10cm/68a5bc63e5584a0f62bfa195.tif
skipped (already exists): data/cog_10cm/68a5bc70e5584a0f62bfa1ce.tif
skipped (already exists): data/cog_10cm/68a78c9e27e991e401df2876.tif
skipped (already exists): data/cog_10cm/68a78caf27e991e401df289b.tif
skipped (already exists): data/cog_10cm/68a78cb427e991e401df28a2.tif
skipped (already exists): data/cog

In [8]:
TILE = 640                  # tamanho do azulejo, em pixels
OVERLAP = 40                # sobreposição entre azulejos vizinhos, em pixels
STRIDE = TILE - OVERLAP     # passo: quanto a janela avança a cada azulejo
TILES_DIR = ROOT / "data" / "tiles_10cm"

index_path = tile_cogs(COG10_DIR, TILES_DIR, gsd=None, tile=TILE, overlap=OVERLAP,
                       labels_dir=LABELS_DIR, previews=True)
index = index_with_paths(TILES_DIR)
print(f"{len(index)} azulejos no total; {int(index.labelled.sum())} desses tiles contém anotações")
print(f"Azulejo {TILE} px, sobreposição {OVERLAP} px -> passo de {STRIDE} px = {STRIDE * 0.10:.0f} m no terreno")
print(f"Índice salvo em: {index_path}")

labels: 737 features from /Users/macuser/Documents/UN/projects/webinar-DL/data/cog_10cm/geojson_labelling for 3 scenes


685740ade319ef725278cb5a.tif: 11 tiles


685740b9e319ef725278cb92.tif: 42 tiles


685740c3e319ef725278cbc1.tif: 7 tiles


685740c7e319ef725278cbcf.tif: 19 tiles


68a4a953e50784296730bf0f.tif: 13 tiles


... (8 outputs left out)

68a78cb427e991e401df28a2.tif: 16 tiles


68a78ce127e991e401df2956.tif: 10 tiles


68a78d0527e991e401df29cc.tif: 9 tiles


68c41fddae0fc8076633a74e.tif: 12 tiles
286 tiles from 17 scenes -> /Users/macuser/Documents/UN/projects/webinar-DL/data/tiles_10cm/ (index: tiles_index.csv)
286 azulejos no total; 31 desses tiles contém anotações
Azulejo 640 px, sobreposição 40 px -> passo de 600 px = 60 m no terreno
Índice salvo em: /Users/macuser/Documents/UN/projects/webinar-DL/data/tiles_10cm/tiles_index.csv


In [9]:
MODEL_ID = "facebook/dinov2-small"
PATCH = 14
INPUT_PX = 644            # 46 x 46 patches; o múltiplo de 14 mais próximo de 640
BATCH_SIZE = 32
FORCE_RECOMPUTE = False   # True para refazer os embeddings mesmo que o arquivo exista

EMB_DIR = ROOT / "data" / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)
EMB_FILE = EMB_DIR / "tiles10cm_dinov2s.npz"

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

processor = AutoImageProcessor.from_pretrained(
    MODEL_ID,
    size={"height": INPUT_PX, "width": INPUT_PX},
    do_center_crop=False,
)
model = AutoModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
print(f"{MODEL_ID} carregado | dispositivo: {DEVICE} | patches por azulejo: {(INPUT_PX // PATCH) ** 2}")
print("arquivo de embeddings:", EMB_FILE.relative_to(ROOT))

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

facebook/dinov2-small carregado | dispositivo: mps | patches por azulejo: 2116
arquivo de embeddings: data/embeddings/tiles10cm_dinov2s.npz


In [10]:
class TileDataset(Dataset):
    # Lê cada azulejo (GeoTIFF) e devolve uma imagem RGB
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with rasterio.open(self.paths[i]) as src:
            arr = src.read([1, 2, 3])
        return Image.fromarray(np.moveaxis(arr, 0, -1))

def collate(images):
    return processor(images=images, return_tensors="pt")["pixel_values"]

def extract(paths):
    loader = DataLoader(TileDataset(paths), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, collate_fn=collate)
    out = np.empty((len(paths), model.config.hidden_size), dtype=np.float32)
    row, t0 = 0, time.time()
    for pixels in loader:
        with torch.inference_mode():
            cls = model(pixel_values=pixels.to(DEVICE)).last_hidden_state[:, 0, :].float()
        cls = torch.nn.functional.normalize(cls, dim=1)
        out[row:row + len(cls)] = cls.cpu().numpy()
        row += len(cls)
        print(f"  {row}/{len(paths)} azulejos, {time.time() - t0:.0f} s")
    assert row == len(paths)
    return out

tile_ids = index.tile_id.tolist()
if EMB_FILE.exists() and not FORCE_RECOMPUTE:
    print(f"{EMB_FILE.relative_to(ROOT)} já existe: pulando a extração.")
else:
    t0 = time.time()
    embeddings = extract(index.tile_file.tolist())
    print(f"pronto: {embeddings.shape} em {(time.time() - t0) / 60:.1f} min")
    np.savez_compressed(
        EMB_FILE,
        embeddings=embeddings,
        sample_ids=np.array(tile_ids),
        tile_ids=np.array(tile_ids),
        model_id=np.array(MODEL_ID),
        input_size=np.array(INPUT_PX),
    )
    print(f"salvo em {EMB_FILE.relative_to(ROOT)} ({EMB_FILE.stat().st_size / 1e6:.1f} MB)")

data/embeddings/tiles10cm_dinov2s.npz já existe: pulando a extração.


In [11]:
data = np.load(EMB_FILE)
emb = data["embeddings"]
print("embeddings:", emb.shape, "| modelo:", str(data["model_id"]), "| entrada:", int(data["input_size"]), "px")
print("Normalizado?", bool(np.allclose(np.linalg.norm(emb, axis=1), 1.0, atol=1e-4)))
row_of = {tid: i for i, tid in enumerate(data["tile_ids"])}
assert all(t in row_of for t in tile_ids), "algum azulejo não tem embedding: rode de novo com FORCE_RECOMPUTE = True"
print("azulejos com embedding:", len(tile_ids), "| tamanho do arquivo:", f"{EMB_FILE.stat().st_size / 1e6:.1f} MB")

embeddings: (286, 384) | modelo: facebook/dinov2-small | entrada: 644 px
Normalizado? True
azulejos com embedding: 286 | tamanho do arquivo: 0.4 MB


In [12]:
# Baixar o arquivo de embeddings
if IN_COLAB:
    from google.colab import files
    files.download(str(EMB_FILE))
else:
    print("Arquivo pronto em:", EMB_FILE)

Arquivo pronto em: /Users/macuser/Documents/UN/projects/webinar-DL/data/embeddings/tiles10cm_dinov2s.npz
